# Data Mining — Predicting Political Orientation from Sejm Speeches

**Goal:** given an MP speech transcript, predict whether it is **left / center / right** (and optionally the parliamentary club).

**Data:** 9th term of the Polish Sejm (2019–2023), API `api.sejm.gov.pl`

**Pipeline:** fetch → clean → label → member-wise split → TF-IDF + Logistic Regression

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, f1_score

DATA = Path("../../data/term9")
SPLITS = DATA / "splits"
MODELS = Path("../../models")

plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True, "grid.alpha": 0.25})
SIDE_ORDER = ["left", "center", "right"]

## 1. Dataset after cleaning and labeling

In [ ]:
df = pd.read_csv(DATA / "dataset_labeled.csv")
df["text_len"] = df["text"].astype(str).str.len()
df["date"] = pd.to_datetime(df["date"])

print(f"Speeches: {len(df):,}")
print(f"MPs:      {df['member_id'].nunique()}")
print(f"Period:   {df['date'].min().date()} – {df['date'].max().date()}")
print(f"Clubs:    {', '.join(sorted(df['club'].unique()))}")
df.head(3)

### Club → orientation mapping

| side | clubs |
|------|--------|
| left | Lewica, LD |
| right | PiS, Konfederacja, Kukiz15, PS |
| center | KO, KP, Polska2050 |

`niez.` was dropped (no clear orientation label).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

club_counts = df["club"].value_counts()
club_counts.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Speeches by club")
axes[0].set_xlabel("Club")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

side_counts = df["side"].value_counts().reindex(SIDE_ORDER)
side_counts.plot(kind="bar", ax=axes[1], color=["#4C78A8", "#F58518", "#E45756"])
axes[1].set_title("Speeches by orientation (side)")
axes[1].set_xlabel("Orientation")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "count": side_counts,
    "share_%": (side_counts / len(df) * 100).round(1),
}))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df.loc[df["text_len"] <= 5000, "text_len"], bins=50, color="steelblue")
ax.axvline(300, color="crimson", linestyle="--", label="cleaning threshold (300 chars)")
ax.set_title("Speech length distribution (≤ 5000 characters)")
ax.set_xlabel("Characters")
ax.set_ylabel("Number of speeches")
ax.legend()
plt.tight_layout()
plt.show()

df["text_len"].describe().round(1).to_frame("text_len")

## 2. Train / val / test split — by MP (`member_id`)

A random row-wise split would leak an MP's speaking style into training and test.  
We therefore split **people**, not individual speeches.

In [ ]:
train = pd.read_csv(SPLITS / "train.csv")
val = pd.read_csv(SPLITS / "val.csv")
test = pd.read_csv(SPLITS / "test.csv")

split_summary = pd.DataFrame([
    {"split": "train", "speeches": len(train), "members": train["member_id"].nunique()},
    {"split": "val", "speeches": len(val), "members": val["member_id"].nunique()},
    {"split": "test", "speeches": len(test), "members": test["member_id"].nunique()},
])
display(split_summary)

ids = {
    "train": set(train["member_id"]),
    "val": set(val["member_id"]),
    "test": set(test["member_id"]),
}
print("Overlap member_id train∩val:", len(ids["train"] & ids["val"]))
print("Overlap member_id train∩test:", len(ids["train"] & ids["test"]))
print("Overlap member_id val∩test:", len(ids["val"] & ids["test"]))

fig, ax = plt.subplots(figsize=(7, 4))
side_by_split = pd.DataFrame({
    "train": train["side"].value_counts(),
    "val": val["side"].value_counts(),
    "test": test["side"].value_counts(),
}).reindex(SIDE_ORDER)
side_by_split.plot(kind="bar", ax=ax)
ax.set_title("Side distribution across splits")
ax.set_xlabel("Orientation")
ax.set_ylabel("Number of speeches")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 3. Baseline model: TF-IDF + Logistic Regression

- TF-IDF vectorization (unigrams + bigrams)
- classifier with `class_weight="balanced"` (class imbalance)
- main task: `side` ∈ {left, center, right}

In [ ]:
side_model = joblib.load(MODELS / "side_tfidf.joblib")

y_true = test["side"]
y_pred = side_model.predict(test["text"])

acc = (y_pred == y_true).mean()
macro = f1_score(y_true, y_pred, average="macro", labels=SIDE_ORDER)
print(f"TEST accuracy: {acc:.3f}")
print(f"TEST macro F1: {macro:.3f}\n")
print(classification_report(y_true, y_pred, labels=SIDE_ORDER, digits=3))

fig, ax = plt.subplots(figsize=(5.5, 5))
cm = confusion_matrix(y_true, y_pred, labels=SIDE_ORDER)
ConfusionMatrixDisplay(cm, display_labels=SIDE_ORDER).plot(
    ax=ax, cmap="Blues", colorbar=False
)
ax.set_title("Confusion matrix — side (test)")
plt.tight_layout()
plt.show()

## 4. Live demo — classifying example speeches

In [ ]:
examples = [
    "Panie Marszałku! Wysoka Izbo! Rząd niszczy niezależność sądów i wolne media. Domagamy się powrotu do praworządności oraz silnej pozycji Polski w Unii Europejskiej.",
    "Panie Marszałku! Wysoka Izbo! Bronimy polskiej rodziny, suwerenności narodu i tradycyjnych wartości. Nie godzimy się na dyktat Brukseli.",
    "Panie Marszałku! Wysoka Izbo! Potrzebujemy nowoczesnej gospodarki, inwestycji w odnawialne źródła energii i odpowiedzialnej polityki publicznej ponad podziałami.",
]

proba = side_model.predict_proba(examples)
classes = list(side_model.classes_)

rows = []
for text, pred, p in zip(examples, side_model.predict(examples), proba):
    rows.append({
        "text": text[:90] + "...",
        "prediction": pred,
        **{c: round(float(v), 3) for c, v in zip(classes, p)},
    })

display(pd.DataFrame(rows))

In [ ]:
# Paste your own speech and run this cell:
my_text = """
Panie Marszałku! Wysoka Izbo! ...
""".strip()

if my_text and not my_text.endswith("..."):
    pred = side_model.predict([my_text])[0]
    probs = dict(zip(side_model.classes_, side_model.predict_proba([my_text])[0]))
    print(f"Prediction: {pred}")
    for label, value in sorted(probs.items(), key=lambda x: -x[1]):
        print(f"  {label}: {value:.3f}")
else:
    print("Fill in the my_text variable with a speech.")

## 5. Takeaways

- A working baseline can predict political orientation from speech text.
- Test accuracy ≈ **0.72**, macro F1 ≈ **0.66**.
- **Right** is recognized best; **left** is weakest (fewer examples + overlap with center).
- Member-wise splitting makes the evaluation more trustworthy.

**Limitations:** club-based labels ≠ full ideology; procedural speeches; strong class imbalance.

**Next steps:** stronger model (Polish transformer), error analysis, optional club classifier.

CLI outside the notebook:
```bash
uv run predict-side "speech text..."
```